# FVF vs. `ON_TARGET_THRESHOLD_DVA`

`compare_fvf_types.ipynb` picked estimator C (selection hazard) at a single threshold,
`ON_TARGET_THRESHOLD_DVA = 1.75`. That constant enters all three estimators in more than one place - it decides
which fixations count as on-target, and therefore which targets count as foveated, where each target's risk
set ends, and which approaches count as peripheral. So the FVF estimates are not obviously independent of it.

This notebook re-runs all three estimators while sweeping the threshold over `np.arange(0.25, 2.6, 0.25)` and
plots FVF against threshold. The question is whether FVF varies with threshold size and, if it does, where it
stabilizes - a plateau would say the estimate reflects a real perceptual quantity rather than the value of the
constant it was derived under.

In [ ]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.io as pio

import config as cnfg
from analysis.helpers.read_data import load_data
from analysis.fvf.fvf import (
    estimate_by_foveation_falloff, estimate_by_launch_distance, estimate_by_selection_hazard,
)

pio.renderers.default = 'notebook'      # 'notebook' or 'browser'

DEFAULT_THRESHOLD = cnfg.ON_TARGET_THRESHOLD_DVA
THRESHOLDS = np.arange(0.25, 2.6, 0.25)

### Read data

Raw fixation-to-target distances don't depend on the threshold - only `on_target`/hazard-curve computation
does - so load once and re-run the estimators per threshold without reloading the pipeline.

In [ ]:
loaded_data = load_data(cnfg.OUTPUT_PATH)
dists = loaded_data.fixation_target_dists
del loaded_data

### Sweep

Pooled FVF per estimator, per threshold. Warnings from A's censoring (expected at low thresholds, where the
foveation-falloff curve is even more saturated) are suppressed here and surfaced instead as NaN rows in the
table.

In [ ]:
import warnings

rows = []
for threshold in THRESHOLDS:
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        _, falloff_pooled, _ = estimate_by_foveation_falloff(dists, on_target_threshold_dva=threshold)
    try:
        _, launch_pooled, _ = estimate_by_launch_distance(dists, on_target_threshold_dva=threshold)
    except ValueError:
        launch_pooled = np.nan
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        _, hazard_pooled, _ = estimate_by_selection_hazard(dists, on_target_threshold_dva=threshold)
    rows.append(dict(
        threshold=threshold,
        foveation_falloff=falloff_pooled,
        launch_distance=launch_pooled,
        selection_hazard=hazard_pooled,
    ))

sweep = pd.DataFrame(rows)
sweep

### Plot: FVF vs. threshold

Looking for a plateau - a range of thresholds over which the estimate barely moves - which would say the FVF
reflects a real perceptual quantity rather than tracking the constant it was computed under. The dotted
vertical line marks the threshold used everywhere else in the pipeline.

In [ ]:
fig = go.Figure()
for i, (col, label) in enumerate([
    ('foveation_falloff', 'A - foveation falloff'),
    ('launch_distance', 'B - launch distance'),
    ('selection_hazard', 'C - selection hazard'),
]):
    fig.add_trace(go.Scatter(
        x=sweep['threshold'], y=sweep[col], mode='lines+markers', name=label,
        line=dict(color=cnfg.get_discrete_color(i, loop=True), width=3),
    ))
fig.add_vline(x=DEFAULT_THRESHOLD, line=dict(dash='dot', width=2, color='grey'))
fig.update_layout(
    title='FVF estimate vs. ON_TARGET_THRESHOLD_DVA',
    xaxis_title='ON_TARGET_THRESHOLD_DVA (DVA)',
    yaxis_title='estimated FVF (DVA)',
    template='plotly_white',
)
fig.show()